In [ ]:
import pandas as pd
import numpy as np
import re
from collections import Counter
df = pd.read_excel('HS52-recycle-import-2024-vải.xlsx')
df_dl=df['Products']
a=df['Products']
df_dl=pd.DataFrame(df_dl,columns=['Products'])

import pandas as pd
import re

# Function to extract percentage values while keeping other details
def extract_percent_values(text):
    """
    Extracts percentage values while filtering out unwanted patterns.
    """
    if pd.isna(text):  # Handle missing values
        return []

    text = text.lower()

    # Extract percentage values
    all_percentages = re.findall(r'\d+\.?\d*%|\d+\.?\d*\s*(?:pct|percent)', text, re.IGNORECASE)

    return all_percentages

# Function to clean only unwanted patterns while keeping the rest intact
def clean_text(text):
    if pd.isna(text):
        return ""

    text = text.lower()

    # Define only the unwanted patterns to remove
    exclude_patterns = [

        r'\bmới\s*\d+%?\b', r'\b\d+%\s*mới\b',  # Matches "mới 98%", "mới 120%", "mới 200%"
        r'\bnew\s*\d+%?\b', r'\b\d+%\s*new\b',  # Matches "new 98%", "new 125%"
        r'\bbrand\s*\d+%?\b', r'\b\d+%\s*brand\b',  # Matches "brand 98%", "98% brand"
        #r'\bsample\s*\d+%?\b', r'\b\d+%\s*sample\b',  # Matches "sample 98%", "98% sample"
        r'\bitem\s*\d+%?\b', r'\b\d+%\s*item\b',  # Matches "item 98%", "98% item"
        r'\bhàng\s*mới\s*\d+%?\b', r'\b\d+%\s*hàng\s*mới\b',  # Matches "hàng mới 98%", "98% hàng mới"
        r'\b\d+%\s*[\#&><?@!*]',  # Removes "98%" if followed by special characters
        r'\(\s*\+\-\s*\d+%\)', r'\+\-\s*\d+%',  # Matches "(+-5%)" or "+-5%"
        r'\(\s*\+/\-\s*\d+%\)', r'\+/\-\s*\d+%',  # Matches "(+/-5%)" or "+/-5%"
        #r'\.\s*\d+%\s*[#><?@&.*]'  # Matches ". 98% .", ". 98% $"
        r'(?<!\d)\.\s*\d+%\s*[#><?@&.*]'
    ]



    for pattern in exclude_patterns:
        text = re.sub(pattern, '', text, flags=re.IGNORECASE)

    # Remove extra spaces caused by removals
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# Apply functions
df_dl["ProductList"] = df_dl["Products"].apply(clean_text)
df_dl["Extracted_Percentages"] = df_dl["ProductList"].apply(extract_percent_values)

# Function to get position matching
def get_position_matching(row):
    """
    Tìm vị trí của các giá trị phần trăm đã trích xuất trong văn bản, đảm bảo khớp chính xác.
    Xử lý tất cả các lần xuất hiện, bao gồm cả trùng lặp, đồng thời đảm bảo các vị trí được nhận diện đúng.
    """
    extracted_percentages = row['Extracted_Percentages']  # List of extracted percentages
    if not extracted_percentages:
        return []

    text = str(row['ProductList']).lower()  # Convert text to lowercase for matching
    matches = []
    used_positions = set()

    # Ensure we match extracted percentages **in order**
    for percent in extracted_percentages:
        # Regex to find exact standalone percentage
        pattern = rf'(?<!\d){re.escape(percent)}(?!\d)'
        match_positions = [m.start() for m in re.finditer(pattern, text)]

        # Assign positions sequentially without duplicating previous matches
        for pos in match_positions:
            if pos not in used_positions:  # Avoid duplicate matches
                matches.append(pos)
                used_positions.add(pos)
                break  # Stop after finding the first valid match

    return sorted(matches)  # Return positions in order

# Apply function to DataFrame
df_dl['position_matching'] = df_dl.apply(get_position_matching, axis=1)
# Function to extract text using provided positions
def extract_text_from_row(row):
    text = row["ProductList"]
    positions = row["position_matching"]  # Predefined list of positions

    if not positions:
        return ""  # Return empty if no positions exist

    # Convert text into word tokens
    words = re.findall(r'\S+', text)
    word_positions = [text.index(word) for word in words]

    # Validate min and max positions within word_positions range
    min_pos = min(positions)
    max_pos = max(positions)

    # Find word indices corresponding to min_pos and max_pos safely
    min_word_idx = next((i for i, pos in enumerate(word_positions) if pos >= min_pos), 0)
    max_word_idx = next((i for i, pos in enumerate(word_positions) if pos >= max_pos), len(words) - 1)

    # Extract range from left window to right window
    left_window_idx = max(0, min_word_idx - 3)  # 2 words before min_pos
    right_window_idx = min(len(words), max_word_idx + 4)  # 2 words after max_pos

    # Extract and return the text within range
    return " ".join(words[left_window_idx:right_window_idx])

# Apply function to each row in the DataFrame
df_dl["extracted_text"] = df_dl.apply(extract_text_from_row, axis=1)
choose_list = ['xơ nhân tạo', 'polyester', 'cotton', 'recycle', 'organic', 'nylon', 'nylon6', 'rayon', 'modal', 'poly', 'polyamide', 'elastane',
               'chemically', 'mechanically', 'viscose', 'acrylic', 'generic', 'thermoplastic', 'cationic', 'polyethylene',
               'cashmere', 'recycled', 'lyra', 'eco', 'matta', 'tencel', 't400', 'pp', 'polypropylen', 'polyurethane','dye',
               'core spun yarrn', 'rec', 'regenerative', 'post consumer', 'merino', 'elastic', 'spandex', 'triacetate',
               'span', 'metallic', 'elastomultiester', 'lyocell', 'ecovero', 'post-consumer', 'taffeta', 'lycra', 'bci','linen',
               'brood', 'protein', 'pa6', 'wool', 'polyethyene', 'acetate', 'spdx', 'other','elastan', 'lurex', 'epe', 'hemp',
               'pes', 'silk', 'birla', 'livaeco', 'elasthan', 'spandexdún', 'elasterell', 'rumble', 'recyclepolyester', 'elastaned',
               'elasta','elast','elastance', 'spanex', 'elasthanne', 'vi', 'ea', 'static', 'biconstituent', 'grey', 'pu', 'spx',
               'polyeser', 'vicose', 'ployeser', 'cly', 'rec', 'polyurethan', 'tpu', 'spande', 'elastimultiester', 'elanstane',
               'fiber','fibers', 'metallised', 'elstance', 'elastanedcdp', 'elastanes', 'el', 'elstane', 'organiccotton', 'biconstituent',
               'metallised', 'polye', 'lyocel', 'polyeste', 'sapndex', 'viscosed', 'bemberg', 'visecose', 'vicoseb','cupro', 'vie',
               'coton', 'truehemp', 'loycell', 'ployester', 'pa', 'gec', 're', 'ly', 'coolmax', 'co', 'acr', 'vis', 'mono', 'bamboo',
               'long', 'staple', 'model', 'pol', 'polyvinyl', 'pva', 'staple', 'polyamid','line', 'thermo','bông',
               'r', 'oc', 'rpes', 'ctn', 'poliester' , 'poliammide', 'metallized', 'spandec', 'po', 'freefit', 'xơ' , 'stape', 'repreve', 'filamment',
               'li', 'ten', 'cotto', 'pe', 'xla', 'cellulose', 'lino', 'ester', 'cttn', 'lycell', 'piw', 'ecolycra', 'plyester', 'elaspan',
               'blackrecyclepolyeste', 'strech','polyuretane', 'n', 'freffit','polyeter','ep', 'elastolefine', 'elas','rr','polyamise','elanstic',
               'soandex', 'cd', 'tel', 'polysester', 'cottotn', 'spadnex','rp', 'paper', 'viscorayon', 'polyestere', 'polyurethane','kevlar','noex','antistatic',
               'sandex','elasstanem']
# Convert choose_list to lowercase for case-insensitive matching
choose_set = set(word.lower() for word in choose_list)


def filter_text_preserve_order(text, choose_set):
    # 1. Chuyển dấu phẩy thành dấu chấm trong phần trăm (cả dấu phẩy đơn lẻ trong số)
    text = re.sub(r'(\d),(\d+)%', lambda m: f"{m.group(1)}.{m.group(2)}%", text)
    text = re.sub(r'(\d),(\d+)', lambda m: f"{m.group(1)}.{m.group(2)}", text)

    # 2. Thêm khoảng trắng giữa chữ và số phần trăm (như "c100%" → "c 100%" hoặc "c20.7%" → "c 20.7%")
    text = re.sub(r'([a-zA-Z])(\d+[.,]?\d*%)', r'\1 \2', text)

    # 3. Thêm khoảng trắng giữa số phần trăm và chữ (ví dụ: "100%cotton" → "100% cotton")
    text = re.sub(r'(\d+[.,]?\d*%)([a-zA-Z])', r'\1 \2', text)

    # 4. Thêm khoảng trắng giữa chữ và số (nếu còn dính liền)
    text = re.sub(r'(?<=[a-zA-Z])(?=\d)', ' ', text)

    # 5. Tokenize: lấy ra số %, chữ, hoặc dấu ngoặc
    tokens = re.findall(
        r'\d+\.\d+%|\d+%|\d+\.\d*\s*(?:pct|percent)|\d+\s*(?:pct|percent)|\b\w+\b|\(|\)',
        text,
        re.IGNORECASE
    )

    # 6. Lọc token
    filtered_tokens = [
        token for token in tokens
        if token.lower() in choose_set
        or re.match(r'\d+\.\d+%|\d+%|\d+\.\d*\s*(?:pct|percent)|\d+\s*(?:pct|percent)', token, re.IGNORECASE)
        or token in "()"
    ]

    return " ".join(filtered_tokens)


# Apply function to each row in the DataFrame
df_dl["filtered_text"] = df_dl["extracted_text"].apply(lambda x: filter_text_preserve_order(x, choose_set))
# Function to ensure '(' always has a matching ')' and keeps valid pairs correctly
def balance_parentheses(text):
    """
    Ensures that every '(' has a matching ')', removing only redundant ones.
    Valid pairs (containing text) are preserved, and misplaced words are corrected.
    """
    open_count = text.count('(')
    close_count = text.count(')')

    # Ensure balanced parentheses by removing excess unmatched ones
    while open_count > close_count:
        text = text[::-1].replace('(', '', 1)[::-1]  # Remove the last unmatched '('
        open_count -= 1

    while close_count > open_count:
        text = text.replace(')', '', 1)  # Remove the first unmatched ')'
        close_count -= 1

    # Ensure valid parentheses with content remain and stay in order
    text = re.sub(r'\(\s*([\w\d%]+.*?)\s*\)', r'(\1)', text)  # Keep valid content inside ()

    # Fix misplaced words outside parentheses (ensures spacing is maintained correctly)
    text = re.sub(r'(\S)\s+\(', r'\1 (', text)  # Ensure '(' follows a word properly
    text = re.sub(r'\)\s+(\S)', r') \1', text)  # Ensure ')' is followed by a space correctly

    # Remove redundant empty parentheses left at the end
    text = re.sub(r'\(\s*\)$', '', text)  # Removes lone '(' at the end

    return text.strip()

# Apply function to each row in the DataFrame
df_dl["balanced_text"] = df_dl["filtered_text"].apply(balance_parentheses)

df_dl=df_dl[['Products','ProductList','filtered_text','balanced_text']]
# Function to remove empty parentheses from text
def remove_empty_parentheses(text):
    """
    Removes empty parentheses '()' from the text while preserving valid ones.
    """
    return re.sub(r'\(\s*\)', '', text).strip()

# Apply function to each row in the DataFrame
df_dl["cleaned_text"] = df_dl["balanced_text"].apply(remove_empty_parentheses)
df_dl=df_dl[['Products','ProductList','cleaned_text']]



In [ ]:
# trường hợp < 100%

df = df_dl

df['total_sum'] = df['cleaned_text'].apply(
    lambda text: int(np.sum([float(num) for num in re.findall(r'\d+\.?\d*', str(text))])) if isinstance(text, str) else 0
)

df_filtered = df[df['total_sum'] < 100].copy()

df_filtered

def extract_percentages(text):
    pattern = r'(\d+(?:\.\d+)?|\d+/\d+)\s?(%|pct)'
    matches = re.findall(pattern, text)

    result = []
    for value, symbol in matches:
        if '/' in value:
            parts = value.split('/')
            for part in parts:
                if part.isdigit():
                    result.append(f"{part}{symbol}")
        else:
            result.append(f"{value}{symbol}")

    return result


def find_percent_positions(text):
    pattern = r'(\d+(?:\.\d+)?|\d+/\d+)\s?(%|pct)'
    return [m.start() for m in re.finditer(pattern, text)]

df_filtered['percent_values'] = df_filtered['ProductList'].apply(extract_percentages)
df_filtered['percent_positions'] = df_filtered['ProductList'].apply(find_percent_positions)
df_filtered_two = df_filtered

df_filtered_two = df_filtered_two[
    df_filtered_two['cleaned_text'].notna() &
    (df_filtered_two['cleaned_text'].str.strip() != "")
]

def extract_text_from_row(row):
    text = row.get("ProductList", "")
    positions = row.get("percent_positions", [])

    # Nếu không có text hợp lệ hoặc positions trống → return ""
    if not isinstance(text, str) or not text.strip() or not positions:
        return ""

    # Tách từ, tính vị trí
    words = re.findall(r'\S+', text)
    word_positions = [text.index(word) for word in words]

    min_pos = min(positions)
    max_pos = max(positions)

    min_word_idx = next((i for i, pos in enumerate(word_positions) if pos >= min_pos), 0)
    max_word_idx = next((i for i, pos in enumerate(word_positions) if pos >= max_pos), len(words) - 1)

    # Trích ngữ cảnh: 4 từ trước và 4 từ sau
    left_window_idx = max(0, min_word_idx - 4)
    right_window_idx = min(len(words), max_word_idx + 4)

    return " ".join(words[left_window_idx:right_window_idx])

df_filtered_two = df_filtered_two[
    df_filtered_two['ProductList'].notna() &
    (df_filtered_two['ProductList'].str.strip() != "")
]

df_filtered_two["cleaned_text"] = df_filtered_two.apply(extract_text_from_row, axis=1)

df_filtered_two.drop(columns=['total_sum', 'percent_values'], inplace=True)

# Danh sách từ khóa
choose_list =['xơ nhân tạo', 'polyester', 'cotton', 'recycle', 'organic', 'nylon', 'nylon6', 'rayon', 'modal', 'poly', 'polyamide', 'elastane',
               'chemically', 'mechanically', 'viscose', 'acrylic', 'generic', 'thermoplastic', 'cationic', 'polyethylene',
               'cashmere', 'recycled', 'lyra', 'eco', 'matta', 'tencel', 't400', 'pp', 'polypropylen', 'polyurethane','dye',
               'core spun yarrn', 'rec', 'regenerative', 'post consumer', 'merino', 'elastic', 'spandex', 'triacetate',
               'span', 'metallic', 'elastomultiester', 'lyocell', 'ecovero', 'post-consumer', 'taffeta', 'lycra', 'bci','linen',
               'brood', 'protein', 'pa6', 'wool', 'polyethyene', 'acetate', 'spdx', 'other','elastan', 'lurex', 'epe', 'hemp',
               'pes', 'silk', 'birla', 'livaeco', 'elasthan', 'spandexdún', 'elasterell', 'rumble', 'recyclepolyester', 'elastaned',
               'elasta','elast','elastance', 'spanex', 'elasthanne', 'vi', 'ea', 'static', 'biconstituent', 'grey', 'pu', 'spx',
               'polyeser', 'vicose', 'ployeser', 'cly', 'rec', 'polyurethan', 'tpu', 'spande', 'elastimultiester', 'elanstane',
               'fiber','fibers', 'metallised', 'elstance', 'elastanedcdp', 'elastanes', 'el', 'elstane', 'organiccotton', 'biconstituent',
               'metallised', 'polye', 'lyocel', 'polyeste', 'sapndex', 'viscosed', 'bemberg', 'visecose', 'vicoseb','cupro', 'vie',
               'coton', 'truehemp', 'loycell', 'ployester', 'pa', 'gec', 're', 'ly', 'coolmax', 'co', 'acr', 'vis', 'mono', 'bamboo',
               'long', 'staple', 'model', 'pol', 'polyvinyl', 'pva', 'staple', 'polyamid','line', 'thermo','bông',
               'r', 'oc', 'rpes', 'ctn', 'poliester' , 'poliammide', 'metallized', 'spandec', 'po', 'freefit', 'xơ' , 'stape', 'repreve', 'filamment',
               'li', 'ten', 'cotto', 'pe', 'xla', 'cellulose', 'lino', 'ester', 'cttn', 'lycell', 'piw', 'ecolycra', 'plyester', 'elaspan',
               'blackrecyclepolyeste', 'strech','polyuretane', 'n', 'freffit','polyeter','ep', 'elastolefine', 'elas','rr','polyamise','elanstic',
               'soandex', 'cd', 'tel', 'polysester', 'cottotn', 'spadnex','rp', 'paper', 'viscorayon', 'polyestere', 'polyurethane','kevlar','noex','antistatic',
               'sandex','elasstanem']

# Convert choose_list thành set để tìm kiếm nhanh hơn
choose_set = set(word.lower() for word in choose_list)

# Hàm lọc văn bản theo danh sách từ khóa
def filter_text_preserve_order(text, choose_set):
    # _100% → 100%
    text = text.replace('_', ' ')

    # 100 % → 100%
    text = re.sub(r'(\d+(?:/\d+)?(?:\.\d+)?)\s+%', r'\1%', text)

    # 30& → 30%
    text = text.replace('&', '%')

    # 100 % → 100%
    text = re.sub(r'(\d+)\s*%', r'\1%', text)

    # nhuom96 pct → nhuom 96pct
    text = re.sub(r'([a-zA-Z]+)(\d+)\s*pct', r'\1 \2pct', text)

    # xử lý dấu phẩy thành dấu chấm trong phần trăm: 69,3% → 69.3%
    text = re.sub(r'(\d+),(\d+)%', lambda m: f"{m.group(1)}.{m.group(2)}%", text)

    # chuẩn hóa mọi dấu phẩy trong số → dấu chấm (tránh nhầm 55,8% → 55.8%)
    text = re.sub(r'(\d),(\d+)', lambda m: f"{m.group(1)}.{m.group(2)}", text)

    # 70/60% → 70% 60%
    text = re.sub(r'(\d+(?:\s*/\s*\d+)+)\s*%',
                  lambda m: " ".join(f"{p}%" for p in re.split(r'\s*/\s*', m.group(1))),
                  text)

    # Thêm khoảng trắng giữa chữ và số phần trăm: "abc95.2%" → "abc 95.2%"
    text = re.sub(r'([a-zA-Z])(\d+[.,]?\d*%)', r'\1 \2', text)

    # Thêm khoảng trắng giữa phần trăm và chữ: "95.2%cotton" → "95.2% cotton"
    text = re.sub(r'(\d+[.,]?\d*%)([a-zA-Z])', r'\1 \2', text)

    # Thêm khoảng giữa chữ và số nếu còn dính liền: "cmvn264" → "cmvn 264"
    text = re.sub(r'(?<=[a-zA-Z])(?=\d)', ' ', text)

    # Tokenize: lấy phần trăm, chữ, hoặc dấu ngoặc
    tokens = re.findall(
        r'\d+\.\d+%|\d+%|\d+\.\d*\s*(?:pct|percent)|\d+\s*(?:pct|percent)|\b\w+\b|\(|\)',
        text,
        re.IGNORECASE
    )

    # Lọc kết quả: chỉ giữ token trong choose_set, phần trăm, hoặc dấu ngoặc
    filtered_tokens = [
        token for token in tokens
        if token.lower() in choose_set
        or re.match(r'\d+\.\d*%|\d+%|\d+\.\d*\s*(?:pct|percent)|\d+\s*(?:pct|percent)', token, re.IGNORECASE)
        or token in "()"
    ]

    return " ".join(filtered_tokens)

if 'df_filtered_two' in locals() and not df_filtered_two.empty:
    df_final = df_filtered_two
    df_final["cleaned_text"] = df_final["cleaned_text"].apply(lambda x: filter_text_preserve_order(x, choose_set))
else:
    print("Lỗi: df_filtered_two không tồn tại hoặc rỗng!")
df_final

def balance_parentheses(text):
    open_count = text.count('(')
    close_count = text.count(')')

    while open_count > close_count:
        text = text[::-1].replace('(', '', 1)[::-1]
        open_count -= 1

    while close_count > open_count:
        text = text.replace(')', '', 1)
        close_count -= 1

    text = re.sub(r'\(\s*([\w\d%]+.*?)\s*\)', r'(\1)', text)
    text = re.sub(r'(\S)\s+\(', r'\1 (', text)
    text = re.sub(r'\)\s+(\S)', r') \1', text)
    text = re.sub(r'\(\s*\)$', '', text)

    return text.strip()

def remove_empty_parentheses(text):
    return re.sub(r'\(\s*\)', '', text).strip()

df_final["balanced_text"] = df_final["cleaned_text"].apply(balance_parentheses)


df_final["cleaned_text"] = df_final["balanced_text"].apply(remove_empty_parentheses)

df_final = df_final[["ProductList", "cleaned_text", "percent_positions"]]

if df_final.empty:
    print("DataFrame df_final hiện đang rỗng")
else:
    df_final


df_filtered.update(df_final)

df_filtered_v = df_filtered[df_filtered['cleaned_text'].fillna('').str.strip() == '']
df_filtered_v

def extract_percentages(text):
    matches = []  # Dùng list để giữ nguyên tất cả phần trăm xuất hiện
    text = re.sub(r'(\d+(?:\.\d+)?)\s+%', r'\1%', text)
    # Tìm tất cả các phần trăm có dạng '23%' hoặc '23.5%'
    direct_matches = re.findall(r'\d+\.?\d*\s?%', text)
    matches.extend(direct_matches)

    # Tìm các nhóm có dấu "/" hoặc "+"
    split_percentages = re.findall(r'([\d/+]+)(?:%|)', text)
    for group in split_percentages:
        if "/" in group or "+" in group:
            numbers = [int(num) for num in re.split(r'[/+]', group) if num.isdigit()]
            if sum(numbers) == 100:  # Chỉ thêm nếu tổng = 100%
                for num in numbers:
                    matches.append(f"{num}%")

    return matches

def find_percent_positions(text):
    positions = []

    # Tìm các vị trí phần trăm trong chuỗi
    for match in re.finditer(r'\d+\.\d+\s?%|\d+\s?%', text):
        positions.append(match.start())

    # Tìm các nhóm có dấu "/" và tính các vị trí cho các phần trăm tách ra
    for match in re.finditer(r'([\d/]+)(?:%|)', text):
        if "/" in match.group(1):
            numbers = [int(num) for num in match.group(1).split('/') if num.isdigit()]
            if sum(numbers) == 100:
                start_index = match.start(1)
                current_pos = start_index
                for num in numbers:
                    if current_pos not in positions:  # Tránh trùng vị trí
                        positions.append(current_pos)
                    current_pos += len(str(num)) + 1  # Cập nhật vị trí

    # Sắp xếp theo vị trí trong chuỗi
    return sorted(positions)

# Áp dụng vào DataFrame của bạn
df_filtered_v['percent_values'] = df_filtered_v['ProductList'].apply(extract_percentages)
df_filtered_v['percent_positions'] = df_filtered_v['ProductList'].apply(find_percent_positions)

df_filtered_two = df_filtered_v

df_filtered_two

# Function to extract text using provided positions
def extract_text_from_row(row):
    text = row["ProductList"]
    positions = row["percent_positions"]  # Predefined list of positions

    if not isinstance(text, str) or not text.strip() or not positions:
        return ""

    # Convert text into word tokens
    words = re.findall(r'\S+', text)
    word_positions = [text.index(word) for word in words]

    # Validate min and max positions within word_positions range
    min_pos = min(positions)
    max_pos = max(positions)

    # Find word indices corresponding to min_pos and max_pos safely
    min_word_idx = next((i for i, pos in enumerate(word_positions) if pos >= min_pos), 0)
    max_word_idx = next((i for i, pos in enumerate(word_positions) if pos >= max_pos), len(words) - 1)

    # Extract range from left window to right window
    left_window_idx = max(0, min_word_idx - 4)  # 2 words before min_pos
    right_window_idx = min(len(words), max_word_idx + 4)  # 2 words after max_pos

    # Extract and return the text within range
    return " ".join(words[left_window_idx:right_window_idx])

# Apply function to each row in the DataFrame
df_filtered_two["cleaned_text"] = df_filtered_two.apply(extract_text_from_row, axis=1)

# df_filtered_two.drop(columns=['total_sum', 'percent_values'], inplace=True)

df_filtered_two

choose_list = ['polyester', 'cotton', 'recycle', 'organic', 'nylon', 'rayon', 'modal', 'poly', 'polyamide', 'elastane',
               'chemically', 'mechanically', 'viscose', 'acrylic', 'generic', 'thermoplastic', 'cationic', 'polyethylene',
               'cashmere', 'recycled', 'lyra', 'eco', 'matta', 'tencel', 't400', 'pp', 'polypropylen', 'polyurethane','dye',
               'core spun yarrn', 'rec', 'regenerative', 'post consumer', 'merino', 'elastic', 'spandex', 'triacetate','modal','span',
               'metallic', 'elastomultiester', 'lyocell', 'ecovero', 'post-consumer', 'taffeta', 'recycle', 'lycra', 'bci','linen',
               'brood', 'protein', 'ny', 'sp', 'ploy ciclo']
choose_set = set(word.lower() for word in choose_list)

def filter_text_preserve_order(text, choose_set):
    text = text.replace('_', ' ')
    text = re.sub(r'(\d+)\s*%', r'\1%', text)  # Chuẩn hóa dấu %
     # 100 % => 100%
    text = re.sub(r'(\d+(?:/\d+)?(?:\.\d+)?)\s+%', r'\1%', text)
    # 30& => 30%
    text = text.replace('&', '%')
    # 100 % => 100%
    text = re.sub(r'(\d+)\s*%', r'\1%', text)
    # 100.00% => 100.00%
    text = re.sub(r'([a-zA-Z])(\d+\.\d+%)', r'\1 \2', text)

    text = re.sub(r'([a-zA-Z])(\d+)', r'\1 \2', text)
    text = re.sub(r'(\d)([a-zA-Z])', r'\1 \2', text)
    # nhuom96 pct => 96pct
    text = re.sub(r'([a-zA-Z]+)(\d+)\s*pct', r'\1 \2pct', text)

    text = re.sub(r'([a-zA-Z])(\d+%)', r'\1 \2', text)
    text = re.sub(r'(\d+%)([a-zA-Z])', r'\1 \2', text)
    text = re.sub(r'([a-zA-Zàáảãạâầấậẫăằắặẵèéẻẽẹêềếệễìíỉĩịòóỏõọôồốộỗơờớợỡùúủũụưừứựữỳýỷỹỵđ])(\d+%)', r'\1 \2', text)

    # Bắt định dạng số phần trăm dạng "58/38/4 cotton/polyester/spandex"
    match = re.search(r'(\d+(?:/\d+)+)\s*([a-zA-Z/\- ]*)', text)
    if match:
        numbers = match.group(1)
        materials = match.group(2).replace('/', ' ')  # Chuyển '/' thành khoảng trắng

        percent_list = [int(num) for num in numbers.split('/')]

        if sum(percent_list) == 100:
            percent_str = " ".join(f"{num}%" for num in percent_list)
            text = text.replace(match.group(0), f"{percent_str} {materials}".strip())

    # Chuyển đổi lại "58/38/4%" thành "58% 38% 4%"
    text = re.sub(r'(\d+(?:/\d+)+)%',
                lambda m: " ".join(f"{num}%" for num in m.group(1).split('/')),
                text)
    text = re.sub(r'(\w)(\d+/\d+)', r'\1 \2', text)
    # Regex mới hỗ trợ cả dấu '/' trong từ
    tokens = re.findall(r'\d+\.\d+%|\d+%|\d+\.\d*\s*(?:pct|percent)|\d+\s*(?:pct|percent)|\b[\w/-]+\b|\(|\)', text, re.IGNORECASE)

    filtered_tokens = [
        token for token in tokens
        if token.lower() in choose_set
        or re.match(r'\d+\.\d*%|\d+%|\d+\.\d*\s*(?:pct|percent)|\d+\s*(?:pct|percent)', token, re.IGNORECASE)
        or token in "()"
    ]

    return " ".join(filtered_tokens)

if 'df_filtered_two' in locals() and not df_filtered_two.empty:
    df_final_v = df_filtered_two.copy()
    df_final_v["cleaned_text"] = df_final_v["cleaned_text"].apply(lambda x: filter_text_preserve_order(x, choose_set))
else:
    print("Lỗi: df_filtered_two không tồn tại hoặc rỗng!")
df_final_v

# Function to ensure '(' always has a matching ')' and keeps valid pairs correctly
def balance_parentheses(text):
    """
    Ensures that every '(' has a matching ')', removing only redundant ones.
    Valid pairs (containing text) are preserved, and misplaced words are corrected.
    """
    open_count = text.count('(')
    close_count = text.count(')')

    # Ensure balanced parentheses by removing excess unmatched ones
    while open_count > close_count:
        text = text[::-1].replace('(', '', 1)[::-1]  # Remove the last unmatched '('
        open_count -= 1

    while close_count > open_count:
        text = text.replace(')', '', 1)  # Remove the first unmatched ')'
        close_count -= 1

    # Ensure valid parentheses with content remain and stay in order
    text = re.sub(r'\(\s*([\w\d%]+.*?)\s*\)', r'(\1)', text)  # Keep valid content inside ()

    # Fix misplaced words outside parentheses (ensures spacing is maintained correctly)
    text = re.sub(r'(\S)\s+\(', r'\1 (', text)  # Ensure '(' follows a word properly
    text = re.sub(r'\)\s+(\S)', r') \1', text)  # Ensure ')' is followed by a space correctly

    # Remove redundant empty parentheses left at the end
    text = re.sub(r'\(\s*\)$', '', text)  # Removes lone '(' at the end

    return text.strip()


def remove_empty_parentheses(text):
    """
    Removes empty parentheses '()' from the text while preserving valid ones.
    """
    return re.sub(r'\(\s*\)', '', text).strip()


try:
    # Nếu df_final_v tồn tại thì xử lý
    df_final_v["balanced_text"] = df_final_v["cleaned_text"].apply(balance_parentheses)
    df_final_v = df_final_v[['ProductList', 'cleaned_text', 'balanced_text', 'percent_positions']]

    df_final_v["cleaned_text"] = df_final_v["balanced_text"].apply(remove_empty_parentheses)
    df_final_v = df_final_v[['ProductList', 'cleaned_text', 'percent_positions']]

    if df_final_v.empty:
        print("DataFrame df_final_v hiện đang rỗng.")
    else:
        df_final_v

except NameError:
    print("DataFrame df_final_v chưa được tạo ra")
    df_final_v=df_filtered
df_final_v

df_filtered.update(df_final_v)
# Function to extract text using provided positions
def extract_text_from_row(row):
    text = row["ProductList"]
    positions = row["percent_positions"]  # Predefined list of positions

    if not isinstance(text, str) or not text.strip() or not positions:
        return ""

    # Convert text into word tokens
    words = re.findall(r'\S+', text)
    word_positions = [text.index(word) for word in words]

    # Validate min and max positions within word_positions range
    min_pos = min(positions)
    max_pos = max(positions)

    # Find word indices corresponding to min_pos and max_pos safely
    min_word_idx = next((i for i, pos in enumerate(word_positions) if pos >= min_pos), 0)
    max_word_idx = next((i for i, pos in enumerate(word_positions) if pos >= max_pos), len(words) - 1)

    # Extract range from left window to right window
    left_window_idx = max(0, min_word_idx - 4)  # 2 words before min_pos
    right_window_idx = min(len(words), max_word_idx + 4)  # 2 words after max_pos

    # Extract and return the text within range
    return " ".join(words[left_window_idx:right_window_idx])

df_temp["cleaned_text"] = df_temp.apply(extract_text_from_row, axis=1)

df_final_two = df_temp

df_final_two

# Danh sách từ khóa
choose_list = ['polyester', 'cotton', 'recycle', 'organic', 'nylon', 'rayon', 'modal', 'poly', 'polyamide', 'elastane',
               'chemically', 'mechanically', 'viscose', 'acrylic', 'generic', 'thermoplastic', 'cationic', 'polyethylene',
               'cashmere', 'recycled', 'lyra', 'eco', 'matta', 'tencel', 't400', 'pp', 'polypropylen', 'polyurethane','dye',
               'core spun yarrn', 'rec', 'regenerative', 'post consumer', 'merino', 'elastic', 'spandex', 'triacetate','modal','span',
               'metallic', 'elastomultiester', 'lyocell', 'ecovero', 'post-consumer', 'taffeta', 'recycle', 'lycra', 'bci','linen',
               'recycled polyester','brood', 'protein', '/']

# Convert choose_list thành set để tìm kiếm nhanh hơn
choose_set = set(word.lower() for word in choose_list)

def filter_text_preserve_order(text, choose_set):
    # Xử lý các ký tự đặc biệt
    text = text.replace('_', ' ')  # Thay thế _ thành dấu cách
    text = re.sub(r'(\d+)\s*%', r'\1%', text)  # Làm sạch khoảng trắng giữa số và dấu %
    text = re.sub(r'([a-zA-Z])(\d+\.\d+%)', r'\1 \2', text)  # Đảm bảo số phần trăm có khoảng trắng nếu có dấu .
    text = re.sub(r'([a-zA-Z]+)(\d+)\s*pct', r'\1 \2pct', text)  # Đảm bảo khoảng trắng giữa chữ và số phần trăm (pct)

    # Xử lý trường hợp như "62/64%" => 62% 64%
    text = re.sub(r'(\d+(?:\s*/\s*\d+)+)\s*%',
                  lambda m: " ".join(f"{p}%" for p in re.split(r'\s*/\s*', m.group(1))),
                  text)

    # Kiểm tra các tỷ lệ phân số như "62/64" không phải phần trăm, giữ nguyên chúng
    # Đảm bảo không làm sai đối với các trường hợp không có dấu %
    text = re.sub(r'(\d+/\d+)(?=\s*[^\d%])', r'\1', text)

    # Kiểm tra trường hợp như "93/7% 58/60" để ưu tiên tách phần trăm sau dấu "/"
    text = re.sub(r'(\d+(?:/\d+)+)\s*(\d+%)', lambda m: " ".join([f"{p}%" for p in m.group(1).split('/')]) + " " + m.group(2), text)

    # Lọc các token (số phần trăm và từ khóa)
    tokens = re.findall(r'\d+\.\d+%|\d+%|\d+\.\d*\s*(?:pct|percent)|\d+\s*(?:pct|percent)|\b\w+\b|/|\(|\)', text, re.IGNORECASE)

    # Lọc những token trong danh sách từ khóa hoặc các phần trăm, pct, hoặc dấu ngoặc
    filtered_tokens = [
        token for token in tokens
        if token.lower() in choose_set
        or re.match(r'\d+\.\d*%|\d+%|\d+\.\d*\s*(?:pct|percent)|\d+\s*(?:pct|percent)', token, re.IGNORECASE)
        or token in "()"
    ]

    return " ".join(filtered_tokens)

# Kiểm tra nếu df_filtered tồn tại và không rỗng
if 'df_filtered' in locals() and not df_filtered.empty:
    df_final_two = df_final_two.copy()
    df_final_two["filtered_text"] = df_final_two["cleaned_text"].apply(lambda x: filter_text_preserve_order(x, choose_set))
else:
    print("Lỗi: df_filtered không tồn tại hoặc rỗng!")

df_final_two

def match_percent_material(text):
    # Step 1: Find all the percentages in the correct order
    percentages = re.findall(r'\d+%', text)

    # Step 2: Split the text by "/" to separate the material names
    # First remove the percentage parts from the text to avoid splitting issues
    material_text = re.sub(r'\d+%', '', text)
    materials = material_text.split('/')
    # Remove extra spaces from material parts
    materials = [m.strip() for m in materials]

    # Step 3: Match percentages with materials
    result = []
    for i, material in enumerate(materials):
        if i < len(percentages):
            result.append(f"{percentages[i]} {material}")

    # Step 4: Join the result

    return " ".join(result)
df_final_two["filtered_text"] = df_final_two["filtered_text"].apply(match_percent_material)

df_clean = df_final_two


if 'cleaned_text' in df_clean.columns:
    df_clean = df_clean.drop(columns=['cleaned_text'])

df_clean = df_clean.rename(columns={'filtered_text': 'cleaned_text'})
df_filtered.update(df_clean)
df_filtered = df_filtered[['Products', 'ProductList', 'cleaned_text']]

df.update(df_filtered)
df = df [['Products', 'ProductList', 'cleaned_text']]
def analyze_case_percent(text):
    if not isinstance(text, str):
        return 0

    text = re.sub(r'(\d+)\s*pct', r'\1%', text)

    percentages = re.findall(r'\d+%', text)

    parts = re.split(r'\d+%', text)

    word_group_count = 0

    for part in parts:
        words = re.findall(r'\b\w+\b', part.strip())
        if words:
            word_group_count += 1

    percentage_count = len(percentages)

    if word_group_count == percentage_count:
        return 0
    elif percentage_count > word_group_count:
        return 1
    else:
        return 2

df['Cases'] = df['cleaned_text'].apply(lambda x: analyze_case_percent(str(x) if isinstance(x, (float, int)) else x))

df_filtered_100= df[df['Cases'].apply(lambda x: x == 1)]
df_filtered_100['percent_values'] = df_filtered_100['ProductList'].apply(extract_percentages)
df_filtered_100['percent_positions'] = df_filtered_100['ProductList'].apply(find_percent_positions)

# Function to extract text using provided positions
def extract_text_from_row(row):
    text = row["ProductList"]
    positions = row["percent_positions"]  # Predefined list of positions

    if not positions:
        return ""  # Return empty if no positions exist

    # Convert text into word tokens
    words = re.findall(r'\S+', text)
    word_positions = [text.index(word) for word in words]

    # Validate min and max positions within word_positions range
    min_pos = min(positions)
    max_pos = max(positions)

    # Find word indices corresponding to min_pos and max_pos safely
    min_word_idx = next((i for i, pos in enumerate(word_positions) if pos >= min_pos), 0)
    max_word_idx = next((i for i, pos in enumerate(word_positions) if pos >= max_pos), len(words) - 1)

    # Extract range from left window to right window
    left_window_idx = max(0, min_word_idx - 5)  # 2 words before min_pos
    right_window_idx = min(len(words), max_word_idx + 5)  # 2 words after max_pos

    # Extract and return the text within range
    return " ".join(words[left_window_idx:right_window_idx])

# Apply function to each row in the DataFrame
df_filtered_100["cleaned_text"] = df_filtered_100.apply(extract_text_from_row, axis=1)

# df_filtered_100_case_1.drop(columns=['total_sum', 'percent_values'], inplace=True)

choose_list =['xơ nhân tạo', 'polyester', 'cotton', 'recycle', 'organic', 'nylon', 'nylon6', 'rayon', 'modal', 'poly', 'polyamide', 'elastane',
               'chemically', 'mechanically', 'viscose', 'acrylic', 'generic', 'thermoplastic', 'cationic', 'polyethylene',
               'cashmere', 'recycled', 'lyra', 'eco', 'matta', 'tencel', 't400', 'pp', 'polypropylen', 'polyurethane','dye',
               'core spun yarrn', 'rec', 'regenerative', 'post consumer', 'merino', 'elastic', 'spandex', 'triacetate',
               'span', 'metallic', 'elastomultiester', 'lyocell', 'ecovero', 'post-consumer', 'taffeta', 'lycra', 'bci','linen',
               'brood', 'protein', 'pa6', 'wool', 'polyethyene', 'acetate', 'spdx', 'other','elastan', 'lurex', 'epe', 'hemp',
               'pes', 'silk', 'birla', 'livaeco', 'elasthan', 'spandexdún', 'elasterell', 'rumble', 'recyclepolyester', 'elastaned',
               'elasta','elast','elastance', 'spanex', 'elasthanne', 'vi', 'ea', 'static', 'biconstituent', 'grey', 'pu', 'spx',
               'polyeser', 'vicose', 'ployeser', 'cly', 'rec', 'polyurethan', 'tpu', 'spande', 'elastimultiester', 'elanstane',
               'fiber','fibers', 'metallised', 'elstance', 'elastanedcdp', 'elastanes', 'el', 'elstane', 'organiccotton', 'biconstituent',
               'metallised', 'polye', 'lyocel', 'polyeste', 'sapndex', 'viscosed', 'bemberg', 'visecose', 'vicoseb','cupro', 'vie',
               'coton', 'truehemp', 'loycell', 'ployester', 'pa', 'gec', 're', 'ly', 'coolmax', 'co', 'acr', 'vis', 'mono', 'bamboo',
               'long', 'staple', 'model', 'pol', 'polyvinyl', 'pva', 'staple', 'polyamid','line', 'thermo','bông',
               'r', 'oc', 'rpes', 'ctn', 'poliester' , 'poliammide', 'metallized', 'spandec', 'po', 'freefit', 'xơ' , 'stape', 'repreve', 'filamment',
               'li', 'ten', 'cotto', 'pe', 'xla', 'cellulose', 'lino', 'ester', 'cttn', 'lycell', 'piw', 'ecolycra', 'plyester', 'elaspan',
               'blackrecyclepolyeste', 'strech','polyuretane', 'n', 'freffit','polyeter','ep', 'elastolefine', 'elas','rr','polyamise','elanstic',
               'soandex', 'cd', 'tel', 'polysester', 'cottotn', 'spadnex','rp', 'paper', 'viscorayon', 'polyestere', 'polyurethane','kevlar','noex','antistatic',
               'sandex','elasstanem']

# Loại bỏ trùng lặp (không phân biệt hoa thường)
choose_set = set(word.lower() for word in choose_list)

# Sắp xếp chính xác theo độ dài (từ dài nhất đến ngắn nhất)
choose_set = sorted(choose_set, key=lambda x: len(x), reverse=True)

choose_pattern = r"\b(?:{})\b".format("|".join(re.escape(word) for word in choose_list))

def filter_text_preserve_order_100(text, choose_set):
    if not isinstance(text, str):
        return ""

    text = text.replace('_', ' ')
    text = text.replace('&', '%')
    text = re.sub(r'([a-zA-Z])(\d+\.\d+%)', r'\1 \2', text)
    text = re.sub(r'(\d+)\s*%', r'\1%', text)
    text = re.sub(r'([a-zA-Z]+)(\d+)\s*pct', r'\1 \2pct', text)
    text = re.sub(r'(\d+)(%)([a-zA-Z])', r'\1\2 \3', text)
    # text = re.sub(r'(\D)(\d+%)', r'\1 \2', text)
    text = re.sub(r'([a-zA-Z]+)(\d+)([a-zA-Z]*)', r'\1 \2\3', text)

    text = re.sub(r'(\S)(elastane|cotton|spandex|recycled)', r'\1 \2', text)
    text = re.sub(r'(elastane|cotton|spandex|recycled)(\S)', r'\1 \2', text)

    tokens = re.findall(r'\d+\.\d+%|\d+%|\d+\.\d*\s*(?:pct|percent)|\d+\s*(?:pct|percent)|\b\w+\b|\(|\)', text, re.IGNORECASE)
    filtered_tokens = [
        token for token in tokens
        if token.lower() in choose_set
        or re.match(r'\d+\.\d*%|\d+%|\d+\.\d*\s*(?:pct|percent)|\d+\s*(?:pct|percent)', token, re.IGNORECASE)
        or token in "()"
    ]

    return " ".join(filtered_tokens)


df_filtered_100["cleaned_text"] = df_filtered_100["cleaned_text"].apply(lambda x: filter_text_preserve_order_100(x, choose_set))

# Function to ensure '(' always has a matching ')' and keeps valid pairs correctly
def balance_parentheses(text):
    """
    Ensures that every '(' has a matching ')', removing only redundant ones.
    Valid pairs (containing text) are preserved, and misplaced words are corrected.
    """
    open_count = text.count('(')
    close_count = text.count(')')

    # Ensure balanced parentheses by removing excess unmatched ones
    while open_count > close_count:
        text = text[::-1].replace('(', '', 1)[::-1]  # Remove the last unmatched '('
        open_count -= 1

    while close_count > open_count:
        text = text.replace(')', '', 1)  # Remove the first unmatched ')'
        close_count -= 1

    # Ensure valid parentheses with content remain and stay in order
    text = re.sub(r'\(\s*([\w\d%]+.*?)\s*\)', r'(\1)', text)  # Keep valid content inside ()

    # Fix misplaced words outside parentheses (ensures spacing is maintained correctly)
    text = re.sub(r'(\S)\s+\(', r'\1 (', text)  # Ensure '(' follows a word properly
    text = re.sub(r'\)\s+(\S)', r') \1', text)  # Ensure ')' is followed by a space correctly

    # Remove redundant empty parentheses left at the end
    text = re.sub(r'\(\s*\)$', '', text)  # Removes lone '(' at the end

    return text.strip()

# Apply function to each row in the DataFrame
df_filtered_100["balanced_text"] = df_filtered_100["cleaned_text"].apply(balance_parentheses)

df_filtered_100=df_filtered_100[['ProductList','cleaned_text','balanced_text', 'percent_positions']]
df_filtered_100
def remove_empty_parentheses(text):
    """
    Removes empty parentheses '()' from the text while preserving valid ones.
    """
    return re.sub(r'\(\s*\)', '', text).strip()

# Apply function to each row in the DataFrame
df_filtered_100["cleaned_text"] = df_filtered_100["balanced_text"].apply(remove_empty_parentheses)
df_filtered_100=df_filtered_100[['ProductList','cleaned_text', 'percent_positions']]
# df_dl.to_excel('df_DL.xlsx', engine='openpyxl')
def clean_percentages(text):
    if not isinstance(text, str):
        return text
    while re.match(r'^(\d+%)\s+(\d+%)', text):
        text = re.sub(r'^(\d+%)\s+(\d+%)', r'\2', text)

    while re.search(r'(\d+%)\s+(\d+%)$', text):
        text = re.sub(r'(\d+%)\s+(\d+%)$', r'\1', text)

    text = re.sub(r'^\d+%\s*\((.*)\)', r'(\1)', text)
    text = re.sub(r'\((.*)\)\s*\d+%$', r'(\1)', text)

    text = re.sub(r'(\d{4,})%', '', text)
    return text.strip()

df_filtered_100['cleaned_text'] = df_filtered_100['cleaned_text'].apply(clean_percentages)

filtered_percent_df =df_filtered_100

df_filtered_100 = df_filtered_100[['ProductList', 'cleaned_text']]
df_filtered_100.update(filtered_percent_df)
df.update(df_filtered_100)

df = df[['Products', 'ProductList', 'cleaned_text']]
df['ProductList'] = df['Products']
# Chuyển về chữ thường
df['ProductList'] = df['ProductList'].astype(str).str.lower()
df.to_excel('df_final.xlsx', engine='openpyxl')

